# VAR-Granger simulation analysis

The four-algorithm benchmark contains 300 primary fits per algorithm: 20 seeds × 3 dynamic conditions × 5 representations. The additional c-GC/c-GC* conditioning-depth fits and their separate 5,000-fit static validation grids are sensitivity and method-validation analyses, not extra replicates in this primary comparison.

In [ ]:
from pathlib import Path
import json, os, shlex, subprocess, sys

def find_package_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'calcium-transient-rising-flank'):
            if (candidate / 'pyproject.toml').is_file() and (candidate / 'examples' / 'run_fast_causal_baselines.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate the package root')

PACKAGE_ROOT = find_package_root()
RUNNER_PYTHON = next((str(path) for path in (PACKAGE_ROOT / '.venv/bin/python', PACKAGE_ROOT / '.venv/Scripts/python.exe') if path.is_file()), sys.executable)
RUNNER_ENV = os.environ.copy()
RUNNER_ENV['MPLBACKEND'] = 'Agg'
RUNNER_ENV['PYTHONUNBUFFERED'] = '1'
RUNNER_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, (str(PACKAGE_ROOT / 'src'), RUNNER_ENV.get('PYTHONPATH'))))
sys.path.insert(0, str(PACKAGE_ROOT / 'src'))
from calcium_transient_rising_flank.checkpointing import format_progress

# Environment setup: uv sync --frozen --all-extras
RUN_VAR_GRANGER = True
RUN_CONFOUNDING_SENSITIVITY = True
RUN_HYPOTHESIS_ANALYSIS = True
N_SEEDS = 20
N_STEPS = 1500
REPRESENTATIONS = 'full,deconvolved,rise,fall,fall_residual'
CONDITIONS = 'dynamic_a_noncausal_fall,dynamic_b_hybrid_causal_fall_overlap,dynamic_c_hybrid_causal_fall_kinetic_misspecification'
OUTPUT_ROOT = PACKAGE_ROOT / 'outputs/matched_dynamic_benchmark'
OUTPUT_DIR = OUTPUT_ROOT / 'var-granger'
command = [
    RUNNER_PYTHON, 'examples/run_fast_causal_baselines.py',
    '--dataset', 'simulations', '--algorithm', 'var-granger',
    '--output-dir', str(OUTPUT_DIR), '--representations', REPRESENTATIONS,
    '--simulation-kinds', 'dynamic-extension', '--conditions', CONDITIONS,
    '--max-lag', '1', '--n-seeds', str(N_SEEDS), '--n-steps', str(N_STEPS),
    '--resume',
]
print(f'[notebook] launching: {shlex.join(command)}', flush=True)
if RUN_VAR_GRANGER:
    subprocess.run(command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
state_path = OUTPUT_DIR / 'simulations_var-granger_resume.json'
if state_path.exists():
    state = json.loads(state_path.read_text())
    print(format_progress(int(state.get('completed_units', 0)), max(1, int(state.get('total_units', 1))), label='VAR-Granger simulation fits'))
summary_path = OUTPUT_DIR / 'summary.json'
if summary_path.exists():
    print(summary_path.read_text())
required = [OUTPUT_ROOT / algorithm / filename for algorithm in ('cgc', 'cgc-star', 'pcmciplus', 'var-granger') for filename in ('baseline_rows.csv', 'summary.json')]
analysis_command = [RUNNER_PYTHON, 'examples/analyze_matched_dynamic_benchmark.py', '--input-root', str(OUTPUT_ROOT), '--output-dir', str(OUTPUT_ROOT / 'analysis')]
print(f'[notebook] analysis command: {shlex.join(analysis_command)}', flush=True)
if RUN_VAR_GRANGER and all(path.is_file() for path in required):
    subprocess.run(analysis_command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
else:
    print('[notebook] analysis waits until all four matched algorithms are complete')
CONFOUNDING_ROOT = PACKAGE_ROOT / 'outputs/matched_confounding_benchmark'
confounding_command = [
    RUNNER_PYTHON, 'examples/run_fast_causal_baselines.py',
    '--dataset', 'simulations', '--algorithm', 'var-granger',
    '--output-dir', str(CONFOUNDING_ROOT / 'var-granger'),
    '--representations', REPRESENTATIONS, '--simulation-kinds', 'confounding',
    '--conditions', 'native,shared_observation_noise,latent_common_driver',
    '--max-lag', '1', '--n-seeds', str(N_SEEDS), '--n-steps', str(N_STEPS),
    '--resume',
]
print(f'[notebook] confounding command: {shlex.join(confounding_command)}', flush=True)
if RUN_CONFOUNDING_SENSITIVITY:
    subprocess.run(confounding_command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
required = [CONFOUNDING_ROOT / algorithm / filename for algorithm in ('cgc', 'cgc-star', 'pcmciplus', 'var-granger', 'lpcmci') for filename in ('baseline_rows.csv', 'summary.json')]
confounding_analysis = [RUNNER_PYTHON, 'examples/analyze_matched_confounding_benchmark.py', '--input-root', str(CONFOUNDING_ROOT), '--output-dir', str(CONFOUNDING_ROOT / 'analysis')]
if RUN_CONFOUNDING_SENSITIVITY and all(path.is_file() for path in required):
    subprocess.run(confounding_analysis, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
else:
    print('[notebook] confounding analysis waits for all five matched algorithms')
HYPOTHESIS_DIR = OUTPUT_DIR / 'hypothesis_analysis'
hypothesis_command = [RUNNER_PYTHON, 'examples/analyze_fast_baseline_hypotheses.py', '--method', 'var-granger', '--baseline-dir', str(OUTPUT_DIR), '--output-dir', str(HYPOTHESIS_DIR), '--n-steps', str(N_STEPS), '--n-null', '99', '--n-sweep-seeds', '20']
print(f'[notebook] H1-H4 command: {shlex.join(hypothesis_command)}', flush=True)
if RUN_HYPOTHESIS_ANALYSIS:
    subprocess.run(hypothesis_command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)


## H1 — Kinetic asymmetry and transient characterization

H1 is a shared input-property check. It characterizes event density, fitted decay persistence, rise counts, rise duration, and signal-to-noise before conditional VAR-Granger is evaluated.


In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display
h1 = pd.read_csv(HYPOTHESIS_DIR / 'h1_transient_characterization.csv')
display(h1.round(4))
display(pd.read_csv(HYPOTHESIS_DIR / 'h1_characterization_summary.csv').round(4))
display(Image(filename=str(HYPOTHESIS_DIR / 'h1_transient.png')))

## H2 — Representation-dependent directed recovery

H2 uses the matched dynamic benchmark. All representations retain the physical time axis and use the same seeds, truth targets, and lag-1 design.


In [ ]:
h2 = pd.read_csv(HYPOTHESIS_DIR / 'h2_representation_recovery.csv')
display(h2.round(4))
display(pd.read_csv(HYPOTHESIS_DIR / 'h2_paired_representation_contrasts.csv').round(4))
display(Image(filename=str(HYPOTHESIS_DIR / 'h2_representations.png')))

## H3 — Null and negative-comparator checks

Observed rise input is compared with 99 cyclic shifts, reverse time, and the falling-flank comparator on one fixed chain truth. Only the cyclic shifts form the empirical null distribution; 99 shifts provide adequate resolution after correction across the two prespecified H3 metrics.


In [ ]:
h3 = pd.read_csv(HYPOTHESIS_DIR / 'h3_null_comparators.csv')
display(h3.round(4))
display(pd.read_csv(HYPOTHESIS_DIR / 'h3_empirical_null_summary.csv').round(4))
display(Image(filename=str(HYPOTHESIS_DIR / 'h3_nulls.png')))

## H4 — Noise and frame-rate robustness

H4 varies observation noise and downsampling while preserving the lag-1 fitting rule. Downsampling changes the physical duration represented by one frame and is therefore reported as a robustness stress, not an identical physical-delay estimand.


In [ ]:
h4 = pd.read_csv(HYPOTHESIS_DIR / 'h4_robustness.csv')
display(h4.round(4))
display(pd.read_csv(HYPOTHESIS_DIR / 'h4_robustness_summary.csv').round(4))
display(pd.read_csv(HYPOTHESIS_DIR / 'h4_extreme_contrasts.csv').round(4))
display(Image(filename=str(HYPOTHESIS_DIR / 'h4_sweeps.png')))
display(Markdown((HYPOTHESIS_DIR / 'analysis-report.md').read_text()))
display(Markdown((HYPOTHESIS_DIR / 'stats-appendix.md').read_text()))

## Reviewer audit: signed information and cross-representation falsification

This matched run tests the information lost by rectification, including inhibitory and biphasic rebound edges, and executes the fall-residual-source to rise-target falsification on dynamic-A.


In [ ]:
RUN_REVIEWER_AUDITS = True
REVIEW_ROOT = PACKAGE_ROOT / 'outputs/representation_bias_benchmark/simulations'
review_command = [RUNNER_PYTHON, 'examples/run_representation_bias_benchmarks.py', '--dataset', 'simulations', '--algorithm', 'var-granger', '--output-dir', str(REVIEW_ROOT / 'var-granger'), '--n-seeds', '20', '--n-steps', '1500', '--n-surrogates', '1000', '--resume']
print(f'[notebook] reviewer-audit command: {shlex.join(review_command)}', flush=True)
if RUN_REVIEWER_AUDITS:
    subprocess.run(review_command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
required = [REVIEW_ROOT / method / filename for method in ('cgc', 'cgc-star', 'pcmciplus', 'var-granger') for filename in ('signed_information_rows.csv', 'cross_representation_rows.csv', 'summary.json')]
review_analysis = [RUNNER_PYTHON, 'examples/analyze_representation_bias_benchmarks.py', '--dataset', 'simulations', '--input-root', str(REVIEW_ROOT), '--output-dir', str(REVIEW_ROOT / 'analysis'), '--n-seeds', '20']
if RUN_REVIEWER_AUDITS and all(path.is_file() for path in required):
    subprocess.run(review_analysis, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
else:
    print('[notebook] common reviewer analysis waits for all four methods')
review_summary_path = REVIEW_ROOT / 'analysis/summary.json'
if review_summary_path.is_file():
    print(review_summary_path.read_text())
